In [ ]:
!pip install gymnasium minigrid openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.7/136.7 kB 4.1 MB/s eta 0:00:00


In [ ]:
import os, json, random
import numpy as np
import gymnasium as gym
import minigrid
import imageio.v2 as imageio
from collections import deque

# --- 공통 유틸 ---
def find_goal_pos(env):
    """MiniGrid 내부 grid를 훑어서 실제 goal 좌표를 찾음"""
    grid = env.unwrapped.grid
    w, h = grid.width, grid.height
    for x in range(w):
        for y in range(h):
            obj = grid.get(x, y)
            if obj is not None and getattr(obj, "type", None) == "goal":
                return (x, y)
    return None

def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def epsilon_by_episode(ep, episodes, eps_start=1.0, eps_end=0.10):
    """에피소드에 따른 epsilon decay (지수형 보간)"""
    frac = ep / max(1, episodes - 1)
    return eps_start * ((eps_end / eps_start) ** frac)

# --- Q-learning agent ---
class QLearningAgent:
    def __init__(self, grid_size, action_size, lr=0.2, gamma=0.95):
        self.grid_size = grid_size
        self.action_size = action_size
        self.q = np.zeros((grid_size * grid_size * 4, action_size), dtype=np.float32)
        self.lr = lr
        self.gamma = gamma
        self.epsilon = 1.0

    def get_state(self, env):
        inner = env.unwrapped
        x, y = inner.agent_pos
        d = inner.agent_dir
        return (x * self.grid_size + y) * 4 + d

    def choose_action(self, state, env):
        if np.random.rand() < self.epsilon:
            return env.action_space.sample()
        return int(np.argmax(self.q[state]))

    def update(self, s, a, r, s2):
        # 표준 Q-learning 업데이트
        predict = self.q[s, a]
        target = r + self.gamma * np.max(self.q[s2])
        self.q[s, a] += self.lr * (target - predict)

# --- (Planner) shaping ---
def planner_shaping(prev_pos, curr_pos, waypoint):
    """
    waypoint까지 가까워지면 +1, 멀어지면 -0.5
    (LLM을 매 스텝 호출하지 않도록 shaping은 purely 계산으로 처리)
    """
    d_prev = manhattan(prev_pos, waypoint)
    d_curr = manhattan(curr_pos, waypoint)
    if d_curr < d_prev:
        return 1.0
    if d_curr > d_prev:
        return -0.5
    return 0.0

# --- (LLM Planner) 에피소드당 1회 호출 + 캐싱 ---
from openai import OpenAI

def plan_waypoints_with_llm(
    start_pos,
    goal_pos,
    grid_size=8,
    max_waypoints=1,          # run_training과 호환 유지
    cache=None,
    model="gpt-4o-mini"
):
    """
    LM as Planner:
    - 에피소드 시작에 한 번 호출해서 waypoint(중간 목표)를 생성한다.
    - JSON으로만 출력하게 강제해서 파싱 안정성 확보.
    - 실패하면 fallback(휴리스틱 waypoint)로 대체하고 경고 로그를 남긴다.
    """
    if cache is None:
        cache = {}

    key = (start_pos, goal_pos, grid_size, max_waypoints, model)
    if key in cache:
        return cache[key]

    # waypoint를 '최소 1개' 만들게 유도 (max_waypoints 만큼)
    prompt = f"""
You are a planner for a grid-world agent.
Grid size: {grid_size}x{grid_size}
Start: {start_pos}
Goal: {goal_pos}

Return ONLY valid JSON with field "waypoints": a list of EXACTLY {max_waypoints} intermediate coordinates.
Rules:
- Each coordinate must be within [0,{grid_size-1}].
- Must NOT equal Start or Goal.
- Do NOT include the goal itself.
- Output ONLY JSON.
Example: {{"waypoints":[[3,1]]}}
"""

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role":"user","content":prompt}],
            temperature=0,
        )
        raw = resp.choices[0].message.content.strip()
        data = json.loads(raw)

        wps = [tuple(x) for x in data.get("waypoints", [])]

        # --- 검증 1: 개수 체크 ---
        if len(wps) != max_waypoints:
            raise ValueError(f"LLM returned {len(wps)} waypoints: {wps}")

        # --- 검증 2: start/goal과 같은 좌표 제거(=실패 처리) ---
        for wp in wps:
            if wp == start_pos or wp == goal_pos:
                raise ValueError(f"LLM waypoint invalid (start/goal): {wp}")

        cache[key] = wps
        return wps

    except Exception as e:
        print(f"[Planner WARNING] LLM waypoint failed -> fallback. reason={repr(e)}")

        # fallback(휴리스틱): start와 goal의 중간점 근처를 waypoint로 사용
        fx = (start_pos[0] + goal_pos[0]) // 2
        fy = (start_pos[1] + goal_pos[1]) // 2
        fallback = [(fx, fy)]

        # 만약 중간점이 start/goal과 겹치면, 살짝 옆으로 이동
        if fallback[0] == start_pos or fallback[0] == goal_pos:
            nx = min(grid_size - 1, fx + 1)
            fallback = [(nx, fy)]

        cache[key] = fallback
        return fallback


def run_training(
    mode,
    episodes=300,
    max_steps=200,
    seed=42,
    enable_llm=True,
    alpha_plan=0.5,
    log_every=1,
    pick_strategy="first_success_vs_fastest",
    env_name="MiniGrid-Empty-8x8-v0",
    llm_model="gpt-4o-mini"
):
    """
    mode: "baseline" or "llm_planner"
    enable_llm: True면 planner에서 LLM 호출, False면 절대 호출 안 함
    alpha_plan: total reward = r_env + alpha_plan * r_plan
    log_every: N 에피소드마다 요약 로그 출력
    pick_strategy: 대표 에피소드 선택 규칙
    """

    np.random.seed(seed)
    random.seed(seed)

    env = gym.make(env_name, render_mode=None)
    grid_size = env.unwrapped.width

    agent = QLearningAgent(grid_size=grid_size, action_size=env.action_space.n)

    # LLM 호출 캐시 (start/goal이 같으면 비용 절감 가능)
    llm_cache = {}

    # --- metrics 저장 ---
    metrics = {
        "success": [],
        "steps": [],
        "R_env": [],
        "R_plan": [],
        "R_total": [],
        "unique_positions": [],  # 에피소드 내에서 방문한 서로 다른 (x,y) 개수 = 탐험 범위(coverage)
    }

    # --- GIF용: 대표 에피소드의 "action sequence"만 저장 (프레임은 나중에 replay해서 생성) ---
    picked = {
        # baseline용
        "baseline_first_success_ep": None,
        "baseline_first_success_actions": None,
        "baseline_first_success_seed": None,

        # baseline 실패만 나올 경우 대비 (가장 긴 에피소드)
        "baseline_longest_ep": None,
        "baseline_longest_actions": None,
        "baseline_longest_seed": None,

        # planner용 (가장 빠른 성공)
        "planner_best_success_ep": None,
        "planner_best_success_steps": None,
        "planner_best_success_actions": None,
        "planner_best_success_seed": None,
    }

    for ep in range(episodes):
        ep_seed = seed + ep
        _, _ = env.reset(seed=ep_seed)

        # epsilon decay 적용
        agent.epsilon = epsilon_by_episode(ep, episodes, 1.0, 0.05)

        # 실제 start/goal 좌표
        start_pos = tuple(env.unwrapped.agent_pos)
        goal_pos = find_goal_pos(env)
        if goal_pos is None:
            raise RuntimeError("Goal 위치를 찾지 못했습니다. MiniGrid 파싱 확인 필요.")

        # --- Planner: 에피소드 시작에만 LLM 호출 ---
        if mode == "llm_planner":
            if enable_llm:
                wps = plan_waypoints_with_llm(
                    start_pos, goal_pos,
                    grid_size=grid_size,
                    max_waypoints=1,
                    cache=llm_cache,
                    model=llm_model
                )
            else:
                wps = []  # LLM 비활성화면 waypoint 없음
        else:
            wps = []

        waypoint_queue = list(wps) + [goal_pos]
        wp_idx = 0

        # --- 에피소드 진행 ---
        s = agent.get_state(env)

        visited_positions = set()
        visited_positions.add(tuple(env.unwrapped.agent_pos))

        sum_env, sum_plan, sum_total = 0.0, 0.0, 0.0
        actions_this_ep = []

        terminated = False
        truncated = False

        for t in range(max_steps):
            a = agent.choose_action(s, env)
            actions_this_ep.append(int(a))

            prev_pos = tuple(env.unwrapped.agent_pos)

            _, r_env, terminated, truncated, _ = env.step(a)

            curr_pos = tuple(env.unwrapped.agent_pos)
            visited_positions.add(curr_pos)

            sum_env += float(r_env)

            r_plan = 0.0
            if mode == "llm_planner":
                wp = waypoint_queue[wp_idx]
                r_plan = planner_shaping(prev_pos, curr_pos, wp)

                # waypoint 도달 시 다음 waypoint로 넘어감
                if curr_pos == wp and wp_idx < len(waypoint_queue) - 1:
                    wp_idx += 1

                sum_plan += r_plan

            r_total = float(r_env) + alpha_plan * float(r_plan)
            sum_total += r_total

            s2 = agent.get_state(env)
            agent.update(s, a, r_total, s2)
            s = s2

            if terminated or truncated:
                break

        # --- metrics 기록 ---
        metrics["success"].append(1 if terminated else 0)
        metrics["steps"].append(t + 1)
        metrics["R_env"].append(sum_env)
        metrics["R_plan"].append(sum_plan)
        metrics["R_total"].append(sum_total)
        metrics["unique_positions"].append(len(visited_positions))

        # --- 대표 에피소드 선택 로직 ---
        if mode == "baseline":
            # (1) 첫 성공 에피소드 저장 (baseline.gif 후보)
            if terminated and picked["baseline_first_success_ep"] is None:
                picked["baseline_first_success_ep"] = ep
                picked["baseline_first_success_actions"] = actions_this_ep
                picked["baseline_first_success_seed"] = ep_seed

            # (2) 성공이 아예 없을 때 대비: 가장 긴 에피소드 저장
            if (picked["baseline_longest_ep"] is None) or ((t + 1) > metrics["steps"][picked["baseline_longest_ep"]]):
                picked["baseline_longest_ep"] = ep
                picked["baseline_longest_actions"] = actions_this_ep
                picked["baseline_longest_seed"] = ep_seed

        if mode == "llm_planner":
            # fastest success 저장 (planner.gif 후보)
            if terminated:
                if (picked["planner_best_success_steps"] is None) or ((t + 1) < picked["planner_best_success_steps"]):
                    picked["planner_best_success_steps"] = (t + 1)
                    picked["planner_best_success_ep"] = ep
                    picked["planner_best_success_actions"] = actions_this_ep
                    picked["planner_best_success_seed"] = ep_seed

        # --- 요약 로그 (너무 많으면 log_every로 조절) ---
        if (ep % log_every) == 0:
            print(
                f"[{mode}] EP {ep:03d} | eps={agent.epsilon:.3f} | succ={int(terminated)} | steps={t+1:3d} | "
                f"R_env={sum_env:6.2f} | R_plan={sum_plan:6.2f} | R_total={sum_total:6.2f} | "
                f"uniq_pos={len(visited_positions):3d} | wps={wps}"
            )

    env.close()
    return metrics, picked


In [ ]:
def save_gif_from_actions(env_name, ep_seed, actions, out_path, fps=8):
    """
    학습 중 기록해둔 actions를 그대로 replay해서 GIF 저장
    - 프레임은 여기서만 생성하므로 메모리 부담 낮음
    """
    env = gym.make(env_name, render_mode="rgb_array")
    env.reset(seed=ep_seed)

    frames = []
    frames.append(env.render())

    for a in actions:
        _, _, terminated, truncated, _ = env.step(int(a))
        frames.append(env.render())
        if terminated or truncated:
            break

    env.close()
    imageio.mimsave(out_path, frames, fps=fps)
    print(f"Saved GIF: {out_path} (frames={len(frames)})")


def make_baseline_and_planner_gifs(env_name, baseline_picked, planner_picked):
    # baseline gif 대상 결정
    if baseline_picked["baseline_first_success_actions"] is not None:
        b_seed = baseline_picked["baseline_first_success_seed"]
        b_actions = baseline_picked["baseline_first_success_actions"]
        b_name = "baseline.gif"
        print(f"[GIF] baseline uses FIRST SUCCESS episode: ep={baseline_picked['baseline_first_success_ep']}")
    else:
        b_seed = baseline_picked["baseline_longest_seed"]
        b_actions = baseline_picked["baseline_longest_actions"]
        b_name = "baseline.gif"
        print(f"[GIF] baseline has no success; uses LONGEST episode: ep={baseline_picked['baseline_longest_ep']}")

    # planner gif 대상 결정
    if planner_picked["planner_best_success_actions"] is None:
        raise RuntimeError("Planner 모드에서 성공 에피소드가 없습니다. (episodes/max_steps/alpha 조정 필요)")
    p_seed = planner_picked["planner_best_success_seed"]
    p_actions = planner_picked["planner_best_success_actions"]
    p_name = "planner.gif"
    print(f"[GIF] planner uses FASTEST SUCCESS episode: ep={planner_picked['planner_best_success_ep']}, steps={planner_picked['planner_best_success_steps']}")

    save_gif_from_actions(env_name, b_seed, b_actions, b_name)
    save_gif_from_actions(env_name, p_seed, p_actions, p_name)


In [ ]:
baseline_metrics, baseline_picked = run_training(
    mode="baseline",
    episodes=300,
    max_steps=200,
    seed=42,
    enable_llm=False,      # baseline은 LLM 호출 없음
    alpha_plan=0.0,
    log_every=10
)
print("baseline success_rate =", np.mean(baseline_metrics["success"]))


[baseline] EP 000 | eps=1.000 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  0.00 | R_total=  0.00 | uniq_pos= 10 | wps=[]
[baseline] EP 010 | eps=0.905 | succ=1 | steps=184 | R_env=  0.35 | R_plan=  0.00 | R_total=  0.35 | uniq_pos= 14 | wps=[]
[baseline] EP 020 | eps=0.818 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  0.00 | R_total=  0.00 | uniq_pos=  8 | wps=[]
[baseline] EP 030 | eps=0.740 | succ=1 | steps=182 | R_env=  0.36 | R_plan=  0.00 | R_total=  0.36 | uniq_pos= 15 | wps=[]
[baseline] EP 040 | eps=0.670 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  0.00 | R_total=  0.00 | uniq_pos=  8 | wps=[]
[baseline] EP 050 | eps=0.606 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  0.00 | R_total=  0.00 | uniq_pos=  6 | wps=[]
[baseline] EP 060 | eps=0.548 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  0.00 | R_total=  0.00 | uniq_pos=  5 | wps=[]
[baseline] EP 070 | eps=0.496 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  0.00 | R_total=  0.00 | uniq_pos=  9 | wps=[]
[baseline] EP 08

In [ ]:
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

planner_metrics, planner_picked = run_training(
    mode="llm_planner",
    episodes=300,
    max_steps=200,
    seed=42,
    enable_llm=True,
    alpha_plan=0.5,
    log_every=10
)
print("planner success_rate =", np.mean(planner_metrics["success"]))


[Planner WARNING] LLM waypoint failed -> fallback. reason=RateLimitError("Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}")
[llm_planner] EP 000 | eps=1.000 | succ=0 | steps=200 | R_env=  0.00 | R_plan=  3.50 | R_total=  1.75 | uniq_pos= 14 | wps=[(3, 3)]
[llm_planner] EP 010 | eps=0.905 | succ=0 | steps=200 | R_env=  0.00 | R_plan= 10.50 | R_total=  5.25 | uniq_pos= 14 | wps=[(3, 3)]
[llm_planner] EP 020 | eps=0.818 | succ=0 | steps=200 | R_env=  0.00 | R_plan= 14.50 | R_total=  7.25 | uniq_pos= 23 | wps=[(3, 3)]
[llm_planner] EP 030 | eps=0.740 | succ=1 | steps= 85 | R_env=  0.70 | R_plan= 10.00 | R_total=  5.70 | uniq_pos= 11 | wps=[(3, 3)]
[llm_planner] EP 040 | eps=0.670 | succ=1 | steps=139 | R_env=  0.51 | R_plan=  5.50 

In [ ]:
make_baseline_and_planner_gifs("MiniGrid-Empty-8x8-v0", baseline_picked, planner_picked)

[GIF] baseline uses FIRST SUCCESS episode: ep=10
[GIF] planner uses FASTEST SUCCESS episode: ep=118, steps=12
Saved GIF: baseline.gif (frames=185)
Saved GIF: planner.gif (frames=13)
